# EDA и моделирование — Predictive Maintenance
# EDA and Modeling — Predictive Maintenance

Разведочный анализ данных и подготовка к построению модели бинарной классификации отказов оборудования.  
Exploratory data analysis and preparation for binary failure classification.

## План работы / Work plan

1. **Загрузка данных** — чтение CSV из `data/raw/`.
2. **Базовая информация** — shape, колонки, типы, пропуски.
3. **Первые строки** — быстрый обзор структуры записей.
4. **Матрица корреляции** — heatmap числовых признаков.
5. **Целевая переменная** — распределение `Target` (0 — нет отказа, 1 — отказ).

> Дальнейшие шаги (модели LR, RF, XGBoost) будут добавлены в следующих ячейках ноутбука.

In [ ]:
# Импорты / Imports
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Стиль графиков / Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Загрузка датасета / Dataset loading

In [ ]:
# Путь к данным относительно корня проекта / Path relative to project root
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "predictive_maintenance.csv"

# Загрузка CSV / Load CSV
df = pd.read_csv(DATA_PATH)

print(f"Файл загружен / File loaded: {DATA_PATH.name}")
print(f"Строк / Rows: {len(df):,}")

## 2. Базовая информация о данных / Basic data overview

In [ ]:
# Размерность / Shape
print("=" * 50)
print("Размерность (строки, столбцы) / Shape (rows, columns):")
print(df.shape)

print("\n" + "=" * 50)
print("Названия колонок / Column names:")
print(list(df.columns))

print("\n" + "=" * 50)
print("Типы данных / Data types:")
print(df.dtypes)

print("\n" + "=" * 50)
print("Пропущенные значения / Missing values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_df[missing_df["missing_count"] > 0] if missing.sum() > 0 else "Пропусков нет / No missing values")

print("\n" + "=" * 50)
print("Краткая статистика числовых признаков / Numeric summary:")
display(df.describe())

## 3. Первые строки датасета / First rows preview

In [ ]:
# Просмотр первых 10 записей / View first 10 records
df.head(10)

## 4. Матрица корреляции признаков / Feature correlation matrix

Для корреляции используем только числовые столбцы.  
Only numeric columns are used for the correlation matrix.

In [ ]:
# Выбираем числовые признаки / Select numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

# Heatmap / Тепловая карта
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # верхний треугольник / upper triangle
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Матрица корреляции числовых признаков\nNumeric feature correlation matrix")
plt.tight_layout()
plt.show()

## 5. Распределение целевой переменной / Target variable distribution

В данном датасете бинарная метка отказа хранится в столбце **`Target`** (0 — нет отказа, 1 — отказ).  
In this dataset, the binary failure label is in **`Target`** (0 = no failure, 1 = failure).

In [ ]:
# Целевая переменная / Target variable
target_col = "Target"

# Подсчёт классов / Class counts
class_counts = df[target_col].value_counts().sort_index()
class_pct = (class_counts / len(df) * 100).round(2)

print("Распределение целевой переменной / Target distribution:")
for label, count in class_counts.items():
    label_name = "Target=0 (нет отказа)" if label == 0 else "Target=1 (отказ)"
    print(f"  {label_name}: {count:,} ({class_pct[label]}%)")

print(f"\nДисбаланс классов / Class imbalance ratio: {class_counts[0] / class_counts[1]:.1f} : 1")

In [ ]:
# Визуализация: countplot + pie chart / Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ["Target=0 (нет отказа)", "Target=1 (отказ)"]
colors = ["#4C72B0", "#C44E52"]

# Столбчатая диаграмма / Bar chart
sns.countplot(data=df, x=target_col, ax=axes[0], palette=colors)
axes[0].set_title("Количество наблюдений по Target\nTarget class counts")
axes[0].set_xlabel("Target")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(labels)

# Круговая диаграмма / Pie chart
axes[1].pie(
    class_counts,
    labels=labels,
    autopct="%1.2f%%",
    colors=colors,
    startangle=90,
    explode=(0, 0.05),
)
axes[1].set_title("Доля классов Target\nTarget class proportions")

plt.tight_layout()
plt.show()

In [ ]:
# Типы отказов среди положительного класса / Failure types among positive class
if "Failure Type" in df.columns:
    failure_types = (
        df.loc[df[target_col] == 1, "Failure Type"]
        .value_counts()
    )
    print("Типы отказов (Failure Type) среди случаев с Target=1:")
    print(failure_types)

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(x=failure_types.index, y=failure_types.values, ax=ax, palette="Reds_r")
    ax.set_title("Распределение типов отказов\nFailure type distribution")
    ax.set_xlabel("Failure Type")
    ax.set_ylabel("Count")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()